<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## An Exact Reference for Heuristic Tour Quality

**Verify the optimum of the small instance and measure how far feasible candidates are from it.**

Keep the five-city symmetric distance matrix, the requirement to visit every city once and return, and the total-distance objective. The decision is a discrete visit order. The clock observations, calibration equality, and correction bound do not apply to this delivery system. Complete enumeration provides an independent reference for assessing a heuristic result; it is not itself the heuristic search.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 1 · Choose a complete tour

A vehicle must visit cities $0,1,2,3,4$ once each and return to its starting city. A **tour** is that complete closed route. Shorter total distance means less travel.

Let $C_{ij}$ be the fixed travel distance from city $i$ to city $j$. The source gives no physical unit, so we report **distance units**. All city pairs are connected, and the distances are symmetric: $C_{ij}=C_{ji}$. The numbers are travel costs; we do not assume they are straight-line distances on a map.

| From / to | $0$ | $1$ | $2$ | $3$ | $4$ |
|:---|---:|---:|---:|---:|---:|
| $0$ | 0 | 2 | 9 | 10 | 7 |
| $1$ | 2 | 0 | 8 | 5 | 6 |
| $2$ | 9 | 8 | 0 | 4 | 3 |
| $3$ | 10 | 5 | 4 | 0 | 1 |
| $4$ | 7 | 6 | 3 | 1 | 0 |

Write the decision as $x=(v_0,v_1,v_2,v_3,v_4,v_5)$, where $v_k$ is the city visited at position $k$. Feasibility requires that $v_0,\ldots,v_4$ contain every city exactly once and $v_5=v_0$. The objective is

$$
f(x)=\sum_{k=0}^{4} C_{v_k,v_{k+1}}.
$$

The final term includes the return trip. For example,

$$
f(2,4,3,1,0,2)=3+1+5+2+9=20.
$$

The distance matrix is fixed data. The visit order is the physical decision. Here the raw performance output, total distance, is also the scalar objective. A route with a missing city or return trip is rejected before any score comparison.


In [ ]:
import numpy as np

# Fixed travel distances, copied from the supplied example_1.py
DISTANCES = np.array([
    [0, 2, 9, 10, 7],
    [2, 0, 8, 5, 6],
    [9, 8, 0, 4, 3],
    [10, 5, 4, 0, 1],
    [7, 6, 3, 1, 0],
], dtype=int)
CITY_COUNT = len(DISTANCES)


def evaluate_tour(tour):
    tour = tuple(tour)
    valid_labels = all(isinstance(city, (int, np.integer))
                       and 0 <= city < CITY_COUNT for city in tour)
    feasible = (len(tour) == CITY_COUNT + 1 and valid_labels
                and tour[0] == tour[-1]
                and set(tour[:-1]) == set(range(CITY_COUNT)))
    if not feasible:
        return {"decision": tour, "feasible": False,
                "edge_distances": None, "objective": None}
    edges = np.array([DISTANCES[a, b] for a, b in zip(tour, tour[1:])])
    return {"decision": tour, "feasible": True,
            "edge_distances": edges, "objective": int(edges.sum())}


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 2 · Establish an exact reference for this small instance

A handful of constructed tours does not cover every feasible tour. To find the actual optimum for this tiny instance, fix city 2 as the written start and enumerate all orders of the other four cities.

There are $4!=24$ orders. Reversed tours have the same distance because the matrix is symmetric, so these include both orientations of 12 distinct undirected cycles. Keeping those duplicates is harmless. Every feasible cycle is represented, so the smallest value in this complete enumeration is a **global optimum for this five-city instance**.

Each tile below is the total distance of one complete tour with city 2 fixed as the written start. The 24 values are sorted. Different tours may share a value, and reverse orientations are both included. The two teal tiles attain the minimum of 20 distance units.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/05-1_heuristic_optimization_greedy_and_local_search/assets/exact_enumeration_coverage.png" alt="Fixing city 2 leaves 4 times 3 times 2 times 1, or 24 complete tours. Their sorted total distances are 20, 20, 21, 21, 22, 22, 22, 22, 24, 24, 25, 25, 30, 30, 31, 31, 33, 33, 33, 33, 34, 34, 35, and 35. Teal highlights the two tours attaining 20." width="512" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: #fff;">
</div>


In [ ]:
from itertools import permutations

reference_start = 2
other_cities = [city for city in range(CITY_COUNT) if city != reference_start]
enumerated_results = [
    evaluate_tour((reference_start,) + order + (reference_start,))
    for order in permutations(other_cities)
]
exact_result = min(enumerated_results, key=lambda result: result["objective"])
exact_distance = exact_result["objective"]
print("Complete tours evaluated:", len(enumerated_results))
print("One globally optimal tour:", exact_result["decision"])
print("Verified optimal distance:", exact_distance)


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 3 · Measure candidate quality

The enumeration gives $f^\star=20$. The relative gap of a feasible tour is

$$
\operatorname{gap}(x)=\frac{f(x)-f^\star}{f^\star}\times100\%.
$$

The start-1 greedy tour has gap $(22-20)/20\times100\%=10\%$. The start-3 tour has distance 24 and gap $(24-20)/20\times100\%=20\%$. Taking the nearest available edge at each step does not guarantee a zero gap. For a larger problem we may not know $f^\star$; a best-known feasible value is not automatically the optimum.

For a controlled comparison, nearest-neighbor construction starts at each city in turn. It repeatedly chooses the closest unvisited city, breaks distance ties by the smaller city label, and returns after visiting all five cities. The matrix and requirements stay fixed.


In [ ]:
def nearest_neighbor(start_city):
    if start_city not in range(CITY_COUNT):
        raise ValueError("The starting city must be one of the city labels.")
    remaining = set(range(CITY_COUNT)) - {start_city}
    tour = [start_city]
    trace = []
    while remaining:
        current = tour[-1]
        next_city = min(remaining,
                        key=lambda city: (DISTANCES[current, city], city))
        trace.append({"current": current, "unvisited": tuple(sorted(remaining)),
                      "chosen": next_city,
                      "distance": int(DISTANCES[current, next_city])})
        tour.append(next_city)
        remaining.remove(next_city)
    tour.append(start_city)
    result = evaluate_tour(tour)
    result["trace"] = trace
    return result


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

Each construction result is feasible. Its total distance is compared with the independently enumerated optimum.


In [ ]:
greedy_results = [nearest_neighbor(start) for start in range(CITY_COUNT)]
for start, result in enumerate(greedy_results):
    gap = 100 * (result["objective"] - exact_distance) / exact_distance
    print(f"Start {start}: gap {gap:.1f}%")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

Enumeration checks $(n-1)!$ written tours with a fixed start, where $n$ is the number of cities. Here $4!=24$, while ten cities would require $9!=362{,}880$ written tours. This exact reference is practical because the example is small. A best-known feasible value in a larger problem does not certify the global optimum.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 4 · Source material and method reference

The supplied `Lecture_10_1_Heuristic_Optimization/10_1.pptx` introduces heuristic construction rules. Its `ExampleCode_10_1/example_1.py` supplies the five-city distance matrix, nearest-neighbor rule, and original start at city 2. This notebook adds complete enumeration and relative-gap calculations for the supplied small instance.

Michael A. Trick's [Heuristics](https://mat.tepper.cmu.edu/orclass/integer/node17.html) describes nearest-neighbor tour construction and two-opt tour improvement. The worked calculations use the supplied matrix.
